# TCC — iniciar U-Net ou U-Mamba no Kaggle

Importar este notebook amanhã, selecionar GPU e anexar o dataset **privado**.
Não contém credenciais nem dados. Não acessa o teste para inferência.
Configurações iguais para comparação: 256, quatro bandas, augmentation,
batch 1, seed 42, AdamW 1e-4, BCE+Dice, 100 épocas.
O treino só começa ao mudar START_TRAINING para True. U-Mamba depende de smoke CUDA.
Hoje foram feitos testes locais; a compatibilidade remota ainda será verificada.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import subprocess
import sys

MODEL = "unet"                 # "unet" ou "umamba"
START_TRAINING = False          # mudar para True somente para iniciar amanhã
SETUP_MAMBA = False             # True na primeira preparação da U-Mamba
ALIGN_TORCH = True              # mesma versão PyTorch/CUDA para os dois modelos
EPOCHS = 100                    # orçamento igual entre modelos
PATCH_SIZE = 256
SEED = 42
RESUME_RUN = ""                # pasta que contém invocation.json e last.pt
CODE_REF = "refs/pull/5/head"   # em retomada: usar o commit impresso na primeira execução
DATA_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "tcc-umamba"
OUTPUT = WORK / "tcc_runs"
assert MODEL in ("unet", "umamba")
assert PATCH_SIZE in (256, 512)
WORK.mkdir(parents=True, exist_ok=True)


## 1. Buscar o código e registrar a versão

Internet precisa estar ativada. A revisão do PR é utilizada porque a main ainda
pode conter a versão anterior. Em retomadas, fixe CODE_REF no commit registrado.
Nenhum arquivo de dados ou resultado é apagado por esta célula.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/oguel/tcc-umamba.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Checkout tem alterações locais; preserve-as antes de trocar de versão.")
subprocess.run(["git", "fetch", "origin", CODE_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Código desta execução:", REVISION)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-runtime.txt")], check=True)


## 2. Preparar Mamba somente se selecionada

O PyTorch/CUDA é alinhado à mesma matriz para os dois modelos (2.10.0+cu126),
somente quando a versão instalada difere. U-Net não instala Mamba. Para U-Mamba, marque
SETUP_MAMBA=True na primeira preparação. O instalador exige Linux x86_64 e
combinação Python/wheel compatível, sem compilação local longa. Todos os comandos
de CUDA/treino são processos novos, evitando reutilizar um torch importado antes
da instalação. Uma falha aqui não deve ser contornada misturando versões.


In [ ]:
if ALIGN_TORCH:
    subprocess.run([sys.executable, "-c",
        "from src.models.umamba_runtime import install_torch_stack; "
        "print('PyTorch alterado:', install_torch_stack())"], cwd=REPO, check=True)
if MODEL == "umamba" and SETUP_MAMBA:
    subprocess.run([sys.executable, "-c",
        "from src.models.umamba_runtime import install_prebuilt_colab_stack; "
        "print('PyTorch alterado:', install_prebuilt_colab_stack())"], cwd=REPO, check=True)
else:
    print("Stack preservado. Mamba será verificada no preflight se selecionada.")
subprocess.run([sys.executable, "-c",
    "import platform,torch; print(platform.platform(), torch.__version__, torch.version.cuda); "
    "assert torch.__version__.startswith('2.10.0+cu126'), 'Alinhe PyTorch ao stack fixado'; "
    "assert torch.cuda.is_available(), 'Selecione uma GPU CUDA no painel do Kaggle'; "
    "print(torch.cuda.get_device_name(0))"], cwd=REPO, check=True)


## 3. Localizar os dados sem alterar a entrada

Anexe o conteúdo de dataset_rgi_v1_privado.zip como dataset privado. Se o Kaggle
mantiver somente o ZIP, a célula verifica seu SHA-256 e extrai uma cópia em working.
Se os arquivos já estiverem extraídos, usa diretamente a entrada somente-leitura.
O preflight seguinte verifica os bytes dos 206 pares e a divisão 89/20/20/77.


In [ ]:
image_folders = list(DATA_ROOT.rglob("Imagens_Individuais"))
if not image_folders:
    import hashlib
    import zipfile
    archives = list(DATA_ROOT.rglob("dataset_rgi_v1_privado.zip"))
    if len(archives) != 1:
        raise RuntimeError("Anexe o dataset privado com os GeoTIFFs ou o ZIP preparado.")
    digest = hashlib.sha256()
    with archives[0].open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == "9cae74a47fcc68885f2703a5bf3cd5d9230eadc400a9c5a57c3ed751376d60b4", "ZIP diferente"
    destination = WORK / "dataset_rgi_v1"
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Caminho inválido no ZIP")
        archive.extractall(destination)
    DATA_ROOT = destination
print("Raiz de dados:", DATA_ROOT)


## 4. Auditoria de partida e smoke CUDA

Verifica o hash dos dados, a atribuição espacial congelada e a GPU. A U-Mamba
também precisa executar seus kernels com forward, backward e atualização finita.
Um erro interrompe a sequência. technical_ready significa prontidão técnica,
não desempenho científico validado nem aprovação automática dos dados.


In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PREP = WORK / "preflight" / stamp
PREP.mkdir(parents=True, exist_ok=False)
subprocess.run([sys.executable, "-m", "scripts.preflight_training", "--root", str(DATA_ROOT),
    "--model", MODEL, "--output", str(PREP / "preflight.json")], cwd=REPO, check=True)
command = [sys.executable, "-m", "scripts.smoke_umamba_gpu", "--model", MODEL,
           "--patch-size", str(PATCH_SIZE), "--output", str(PREP / "smoke.json")]
if MODEL == "umamba":
    command.append("--prepare-runtime")
subprocess.run(command, cwd=REPO, check=True)
READY = json.loads((PREP / "preflight.json").read_text())["technical_ready"]
print("Verificação técnica aprovada:", READY)


## 5. Iniciar ou retomar treinamento

Mude START_TRAINING=True na primeira célula. O orçamento é o total de épocas,
inclusive na retomada. RESUME_RUN deve apontar para a pasta restaurada do treino;
o código recusa mudanças de dados, divisão, código, configuração ou PyTorch.
Não reutilize os pesos dos ensaios técnicos como inicialização científica.
Treino usa 89 patches (356 vistas/época em 256), validação usa 20 (80 vistas).
O teste reservado nunca é inferido. Uma interrupção perde no máximo a época
incompleta, desde que last.pt tenha sido preservado fora da sessão efêmera.


In [ ]:
if START_TRAINING:
    preflight = json.loads((PREP / "preflight.json").read_text())
    smoke = json.loads((PREP / "smoke.json").read_text())
    assert READY and preflight["model"] == MODEL and smoke["step_completed"]
    assert smoke["model"] == ("UNet" if MODEL == "unet" else "UMambaEnc_2d")
    assert smoke["input_shape"][-2:] == [PATCH_SIZE, PATCH_SIZE]
    OUTPUT.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", "-m", "scripts.train_emater_rgi",
        "--root", str(DATA_ROOT), "--model", MODEL, "--device", "cuda",
        "--run-mode", "training", "--epochs", str(EPOCHS), "--batch-size", "1",
        "--patch-size", str(PATCH_SIZE), "--augment", "--seed", str(SEED),
        "--train-views-per-image", "4" if PATCH_SIZE == 256 else "1",
        "--num-workers", "0", "--output", str(OUTPUT)]
    if RESUME_RUN:
        command += ["--resume-run", RESUME_RUN]
    with (OUTPUT / (stamp + "_console.log")).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait():
                raise RuntimeError("Treino interrompido/falhou; consulte status.json e last.pt.")
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
else:
    print("Treinamento não iniciado. Mude START_TRAINING=True quando estiver pronto.")


## 6. Preservar saídas e retomar em outra sessão

working é armazenamento da sessão. Antes de encerrá-la, preserve os outputs pelo
Kaggle e/ou baixe este arquivo. last.pt e history.json são gravados a cada época,
mas isso não é backup externo automático. Perda da sessão antes de preservar os
arquivos ainda pode perder os checkpoints. Não envie pesos ao GitHub.
Para retomar: anexe as saídas preservadas, copie a pasta completa de execução para
working, fixe CODE_REF no commit original e configure RESUME_RUN nessa pasta.
A retomada é pela última época, não por best_*.pt.


In [ ]:
if OUTPUT.exists():
    import shutil
    backup = shutil.make_archive(str(WORK / ("backup_tcc_" + stamp)), "zip", root_dir=OUTPUT)
    print("Backup local da sessão:", backup)
    for status in sorted(OUTPUT.glob("*/status.json")):
        print(status.parent.name, status.read_text())
else:
    print("Nenhum treino iniciado; não há checkpoints para copiar.")
